# 01. Tối ưu không ràng buộc — Genetic Algorithm tìm Global Optimum

Notebook nhập hàm mục tiêu $f(x)$ từ bàn phím (không ràng buộc) và dùng
**Genetic Algorithm** tự cài để tìm cực tiểu toàn cục (global optimum),
đối chiếu với **PyGAD** — thư viện GA có sẵn — cả về kết quả lẫn thời
gian chạy.

**Cách dùng:** chạy các cell từ trên xuống (`Run All`). Muốn đổi hàm mục
tiêu / miền tìm kiếm thì sửa form ở cell ①, bấm **Áp dụng bài toán**, rồi
chạy lại từ cell ②.

Hàm mẫu dựng sẵn: **Easom function**
$$f(x, y) = -\cos(x)\cos(y)\, e^{-((x-\pi)^2+(y-\pi)^2)}$$
cực tiểu toàn cục $f(\pi, \pi) = -1$, gần như bằng phẳng ($\approx 0$) ở
mọi nơi khác — một bài toán kinh điển để kiểm tra khả năng tìm ĐÚNG cực
trị TOÀN CỤC thay vì dừng lại ở vùng phẳng xung quanh.

In [1]:
import functools
import re
import time

import numpy as np
import pygad
import sympy
sp = sympy

from sympy.parsing.sympy_parser import (
    parse_expr,
    standard_transformations,
    implicit_multiplication,
    convert_xor,
    function_exponentiation,
)


# ============================================================
# 1. PARSER — giống GA_test.ipynb, chỉ giữ phần cần cho hàm mục tiêu
#    (không có ràng buộc nên bỏ parse_constraint / parse_constraints)
# ============================================================

TRANSFORMATIONS = standard_transformations + (
    implicit_multiplication,
    convert_xor,
    function_exponentiation,
)

# Các hàm / hằng số toán học mà user được phép nhập.
# Đây cũng là các tên KHÔNG được dùng làm tên biến.
LOCAL_DICT = {
    "sin": sp.sin,
    "cos": sp.cos,
    "tan": sp.tan,
    "asin": sp.asin,
    "acos": sp.acos,
    "atan": sp.atan,
    "sinh": sp.sinh,
    "cosh": sp.cosh,
    "tanh": sp.tanh,
    "exp": sp.exp,
    "log": sp.log,
    "ln": sp.log,
    "sqrt": sp.sqrt,
    "abs": sp.Abs,
    "Abs": sp.Abs,
    "pi": sp.pi,
    "e": sp.E,
    "E": sp.E,
}

GLOBAL_DICT = {
    "Symbol": sp.Symbol,
    "Integer": sp.Integer,
    "Float": sp.Float,
    "Rational": sp.Rational,
}

RESERVED_NAMES = set(LOCAL_DICT)


def parse_math_expr(text):
    """Parse biểu thức toán học tự nhiên, ví dụ: x^2 + y^2, 2x + 3y, sin(x)+cos(y)."""

    text = text.strip()

    if not text:
        raise ValueError("Biểu thức rỗng.")

    try:
        return parse_expr(
            text,
            local_dict=LOCAL_DICT,
            global_dict=GLOBAL_DICT,
            transformations=TRANSFORMATIONS,
            evaluate=True,
        )

    except Exception as error:
        raise ValueError(
            f"Không đọc được biểu thức {text!r}: {error}"
        ) from error


def _natural_sort_key(symbol):
    """Sắp biến theo thứ tự tự nhiên: x1, x2, x10 thay vì x1, x10, x2."""

    parts = re.split(r"(\d+)", symbol.name)

    return [
        (1, int(part), "") if part.isdigit() else (0, 0, part)
        for part in parts
    ]


def _implicit_products(symbols):
    """Suy ra phép nhân ngầm từ tên biến bị dính liền (xem GA_test.ipynb)."""

    don_le = {
        symbol.name
        for symbol in symbols
        if len(symbol.name) == 1 and symbol.name.isalpha()
    }

    thay_the = {}

    for symbol in symbols:
        ten = symbol.name

        if len(ten) < 2 or not ten.isalpha():
            continue

        if all(chu in don_le for chu in ten):
            tich = sp.Integer(1)
            for chu in ten:
                tich *= sp.Symbol(chu)
            thay_the[symbol] = tich

    return thay_the


def build_unconstrained_problem(objective_text):
    """Parse hàm mục tiêu, tự tìm biến quyết định, dựng hàm số để đánh giá."""

    objective_expr = parse_math_expr(objective_text)

    symbols = set(objective_expr.free_symbols)

    # Tách tên dính liền thành phép nhân: '2xy' -> 2*x*y
    thay_the = _implicit_products(symbols)

    if thay_the:
        objective_expr = sp.expand(objective_expr.subs(thay_the))
        symbols = set(objective_expr.free_symbols)

    variables = sorted(symbols, key=_natural_sort_key)

    if not variables:
        raise ValueError("Không tìm thấy biến quyết định.")

    objective_raw = sp.lambdify(variables, objective_expr, modules="numpy")

    def objective(x):
        try:
            value = float(
                np.asarray(objective_raw(*x)).reshape(())
            )

            if np.isfinite(value):
                return value

        except Exception:
            pass

        return np.inf

    return objective_expr, variables, objective


# ============================================================
# 2. GENETIC ALGORITHM CHO TỐI ƯU KHÔNG RÀNG BUỘC
# ============================================================
#
# Khác với GA có ràng buộc (xem GA_test.ipynb): ở đây fitness = f(x)
# trực tiếp, không cần quy tắc khả thi Deb hay ngưỡng epsilon giảm dần.
#
# Miền tìm kiếm [lower, upper] đóng vai trò MIỀN XÁC ĐỊNH của bài toán
# tối ưu toàn cục, nên cá thể được CẮT (clip) về đúng miền này sau mỗi
# phép lai ghép / đột biến — khác GA có ràng buộc, nơi hộp chỉ dùng để
# khởi tạo quần thể chứ không giới hạn cá thể.


# Tham số GA dùng CHUNG cho cả bản tự cài lẫn PyGAD bên dưới, để việc
# so sánh không lệch nhau vì hai bộ tham số khác nhau.
CROSSOVER_RATE = 0.9
MUTATION_RATE = 0.15
MUTATION_SCALE = 0.08
ELITE_SIZE = 2
TOURNAMENT_SIZE = 3


def genetic_algorithm_unconstrained(
    objective,
    bounds,

    population_size=100,
    generations=500,

    crossover_rate=CROSSOVER_RATE,
    mutation_rate=MUTATION_RATE,
    mutation_scale=MUTATION_SCALE,

    elite_size=ELITE_SIZE,
    tournament_size=TOURNAMENT_SIZE,

    seed=42,
):
    """GA mã hóa số thực cho bài toán tối ưu KHÔNG ràng buộc, tìm cực tiểu
    toàn cục của f(x) trong hộp [lower, upper]."""

    rng = np.random.default_rng(seed)

    bounds = np.asarray(bounds, dtype=float)

    lower = bounds[:, 0]
    upper = bounds[:, 1]

    variable_range = upper - lower
    n_variables = len(bounds)

    def clip(pop):
        return np.clip(pop, lower, upper)

    # --------------------------------------------------------
    # Initial population
    # --------------------------------------------------------

    population = clip(
        rng.uniform(lower, upper, size=(population_size, n_variables))
    )

    def evaluate(pop):
        return np.array([objective(individual) for individual in pop])

    # --------------------------------------------------------
    # Tournament selection (theo thứ hạng fitness)
    # --------------------------------------------------------

    def tournament_selection(rank):

        indices = rng.integers(0, population_size, size=tournament_size)

        best_index = indices[np.argmin(rank[indices])]

        return population[best_index].copy()

    # --------------------------------------------------------
    # Blend crossover
    # --------------------------------------------------------

    def crossover(parent1, parent2):

        if rng.random() > crossover_rate:
            return parent1.copy(), parent2.copy()

        alpha = rng.uniform(-0.25, 1.25, size=n_variables)

        child1 = alpha * parent1 + (1 - alpha) * parent2
        child2 = alpha * parent2 + (1 - alpha) * parent1

        return child1, child2

    # --------------------------------------------------------
    # Gaussian mutation
    # --------------------------------------------------------

    def mutate(child):

        mutation_mask = rng.random(n_variables) < mutation_rate

        if np.any(mutation_mask):
            child[mutation_mask] += rng.normal(
                loc=0,
                scale=mutation_scale * variable_range[mutation_mask],
            )

        return child

    # --------------------------------------------------------
    # Evolution
    # --------------------------------------------------------

    history = []

    start_time = time.perf_counter()

    fitness = evaluate(population)

    best_index = int(np.argmin(fitness))
    best_solution = population[best_index].copy()
    best_fitness = fitness[best_index]

    for generation in range(generations):

        order = np.argsort(fitness)

        rank = np.empty(population_size, dtype=np.int64)
        rank[order] = np.arange(population_size)

        current_best = int(np.argmin(fitness))
        if fitness[current_best] < best_fitness:
            best_fitness = fitness[current_best]
            best_solution = population[current_best].copy()

        history.append(best_fitness)

        # Elitism
        new_population = [
            population[i].copy()
            for i in order[:elite_size]
        ]

        # Sinh thế hệ tiếp theo
        while len(new_population) < population_size:

            parent1 = tournament_selection(rank)
            parent2 = tournament_selection(rank)

            child1, child2 = crossover(parent1, parent2)

            new_population.append(mutate(child1))

            if len(new_population) < population_size:
                new_population.append(mutate(child2))

        population = clip(np.asarray(new_population))
        fitness = evaluate(population)

    current_best = int(np.argmin(fitness))
    if fitness[current_best] < best_fitness:
        best_fitness = fitness[current_best]
        best_solution = population[current_best].copy()

    elapsed_time = time.perf_counter() - start_time

    return {
        "x": best_solution,
        "fun": best_fitness,
        "time": elapsed_time,
        "history": history,
        "generations": generations,
        "seed": seed,
    }


# ============================================================
# 3. PYGAD — THƯ VIỆN GA CÓ SẴN, DÙNG LÀM MỐC SO SÁNH
# ============================================================
#
# Dùng LẠI đúng công thức lai ghép (blend crossover) và đột biến
# (Gaussian) của GA tự cài ở trên, qua hàm tùy chỉnh — PyGAD không có
# sẵn hai toán tử này (chỉ có single-point/two-points/uniform/scattered/
# sbx cho crossover, và random/swap/inversion/scramble cho mutation).
# Nhờ vậy khác biệt còn lại chỉ nằm ở KIẾN TRÚC vòng lặp của thư viện —
# PyGAD chọn sẵn một pool `num_parents_mating` cha mẹ mỗi thế hệ (qua
# tournament) rồi lai ghép tuần tự trong pool đó, còn GA tự cài chọn 2
# cha mẹ MỚI qua tournament cho MỖI phép lai — không phải khác biệt về
# công thức toán học của từng toán tử.
#
# PyGAD tối đa hóa fitness, nên fitness = -f(x) để tương đương tối
# thiểu hóa f(x). gene_space={'low','high'} vừa khởi tạo vừa giữ mỗi
# gene trong đúng miền tìm kiếm, giống clip() ở GA tự cài.

def run_pygad(objective, bounds, n_variables, population_size, generations, seed):
    """Chạy PyGAD trên cùng bài toán, trả về kết quả cùng định dạng với
    genetic_algorithm_unconstrained để show_solution/show_comparison dùng chung.

    Cả lai ghép (blend crossover) lẫn đột biến (Gaussian) đều dùng lại
    ĐÚNG công thức của GA tự cài, qua hàm tùy chỉnh truyền vào
    crossover_type / mutation_type — không dùng toán tử mặc định của
    PyGAD, để hai bản cài đặt xử lý cá thể theo cùng một cách."""

    bounds_arr = np.asarray(bounds, dtype=float)
    lower = bounds_arr[:, 0]
    upper = bounds_arr[:, 1]
    variable_range = upper - lower

    rng = np.random.default_rng(seed)

    def fitness_func(ga_instance, solution, solution_idx):
        return -objective(solution)

    def gaussian_mutation(offspring, ga_instance):
        for i in range(offspring.shape[0]):
            mask = rng.random(n_variables) < MUTATION_RATE
            if np.any(mask):
                offspring[i, mask] += rng.normal(
                    0, MUTATION_SCALE * variable_range[mask]
                )
            offspring[i] = np.clip(offspring[i], lower, upper)
        return offspring

    def blend_crossover(parents, offspring_size, ga_instance):
        num_offspring, num_genes = offspring_size
        n_parents = parents.shape[0]
        offspring = np.empty(offspring_size, dtype=float)

        for k in range(num_offspring):
            parent1 = parents[k % n_parents]
            parent2 = parents[(k + 1) % n_parents]

            if rng.random() > CROSSOVER_RATE:
                offspring[k] = parent1.copy()
                continue

            alpha = rng.uniform(-0.25, 1.25, size=num_genes)
            offspring[k] = alpha * parent1 + (1 - alpha) * parent2

        return np.clip(offspring, lower, upper)

    start_time = time.perf_counter()

    ga_instance = pygad.GA(
        num_generations=generations,
        num_parents_mating=max(2, population_size // 2),
        fitness_func=fitness_func,
        sol_per_pop=population_size,
        num_genes=n_variables,
        gene_space=[{"low": b[0], "high": b[1]} for b in bounds],
        parent_selection_type="tournament",
        K_tournament=TOURNAMENT_SIZE,
        crossover_type=blend_crossover,
        mutation_type=gaussian_mutation,
        keep_elitism=ELITE_SIZE,
        random_seed=seed,
        suppress_warnings=True,
    )
    ga_instance.run()

    solution, _fitness, _index = ga_instance.best_solution()
    solution = np.asarray(solution, dtype=float)

    elapsed_time = time.perf_counter() - start_time

    return {
        "x": solution,
        "fun": objective(solution),
        "time": elapsed_time,
    }


print(f"numpy {np.__version__} | pygad {pygad.__version__} | sympy {sympy.__version__}")
print("Tên dành riêng (không dùng làm biến):", ", ".join(sorted(RESERVED_NAMES)))


# ------------------------------------------------------------------
# Hiển thị dạng ký hiệu toán học (LaTeX)
# ------------------------------------------------------------------
from IPython.display import Markdown, display


def _num(value, digits=10):
    """Số dạng LaTeX; chuyển sang ký hiệu khoa học khi quá lớn hoặc quá nhỏ."""
    if not np.isfinite(value):
        return r"\infty" if value > 0 else r"-\infty"
    if value != 0 and (abs(value) >= 1e6 or abs(value) < 1e-4):
        mantissa, exponent = f"{value:.4e}".split("e")
        return mantissa + r" \times 10^{" + str(int(exponent)) + "}"
    return f"{value:.{digits}f}"


def _num_bound(value):
    """Số dạng LaTeX cho cận tìm kiếm — bỏ số 0 thừa: -10 thay vì -10.0000."""
    text = _num(value, 4)
    if "." in text and "times" not in text:
        text = text.rstrip("0").rstrip(".")
    return text


def show_problem(objective_expr, variables, bounds):
    """Phát biểu bài toán tối ưu không ràng buộc và miền tìm kiếm."""
    bien = ", ".join(sympy.latex(v) for v in variables)
    mien = ", \\ ".join(
        f"{_num_bound(b[0])} \\le {sympy.latex(v)} \\le {_num_bound(b[1])}"
        for v, b in zip(variables, bounds)
    )
    display(Markdown(
        "$$\n\\underset{" + bien + r"}{\text{minimize}} \quad f\left("
        + bien + r"\right) = " + sympy.latex(objective_expr) + "\n$$\n\n"
        + "Miền tìm kiếm: $" + mien + "$"
    ))


def show_solution(name, result, variables):
    """Nghiệm tìm được và thời gian chạy."""
    toado = r" \\ ".join(
        sympy.latex(v) + " &= " + _num(x) for v, x in zip(variables, result["x"])
    )

    khoi = [
        "**" + name + "**",
        "",
        r"$$\begin{aligned}" + toado + r"\end{aligned}$$",
        r"$$f^{*} = " + _num(result["fun"]) + r"$$",
        "",
        "| | |",
        "|---|---|",
        "| Thời gian chạy | $" + _num(result["time"], 6) + r"\ \text{s}$ |",
    ]

    display(Markdown("\n".join(khoi)))


def show_statistics(runs):
    """Thống kê qua nhiều lần chạy GA độc lập."""
    gia_tri = np.array([r["fun"] for r in runs if np.isfinite(r["fun"])])

    if len(gia_tri) == 0:
        display(Markdown("*Không lần chạy nào cho giá trị hữu hạn.*"))
        return

    display(Markdown("\n".join([
        "**Thống kê GA qua " + str(len(runs)) + " lần chạy độc lập**",
        "",
        "| | |",
        "|---|---|",
        r"| Tốt nhất | $\min f = " + _num(gia_tri.min()) + "$ |",
        r"| Trung bình | $\bar{f} = " + _num(gia_tri.mean()) + "$ |",
        r"| Tệ nhất | $\max f = " + _num(gia_tri.max()) + "$ |",
        r"| Độ lệch chuẩn | $\sigma = " + _num(gia_tri.std()) + "$ |",
    ])))


def show_comparison(results, variables):
    """Bảng so sánh nhiều phương pháp: f*, thời gian, tọa độ nghiệm.

    `results` là danh sách [(tên, result), ...]."""
    cot_bien = " | ".join("$" + sympy.latex(v) + "$" for v in variables)

    dong = [
        r"| Phương pháp | $f^{*}$ | Thời gian (s) | " + cot_bien + " |",
        "|---|---|---|" + "---|" * len(variables),
    ]

    for ten, r in results:
        toado = " | ".join("$" + _num(x) + "$" for x in r["x"])
        dong.append("| " + ten + " | $" + _num(r["fun"]) + "$ | $"
                    + _num(r["time"], 6) + "$ | " + toado + " |")

    display(Markdown("\n".join(dong)))


numpy 1.26.4 | pygad 3.7.0 | sympy 1.14.0
Tên dành riêng (không dùng làm biến): Abs, E, abs, acos, asin, atan, cos, cosh, e, exp, ln, log, pi, sin, sinh, sqrt, tan, tanh


---
## ① Nhập hàm mục tiêu

Điền vào form rồi bấm **Áp dụng bài toán**.

**Hàm mục tiêu** — `x^2 + y^2` hoặc `x**2 + y**2`, `2x` = `2*x`, hỗ trợ
`sqrt exp log ln abs sin cos tan`.

⚠️ **Tích hai biến phải có dấu `*`**: gõ `2xy` thì `xy` thành một biến mới
chứ không phải `2*x*y`. Viết `2*x*y`. Chương trình sẽ báo lỗi nếu phát hiện.

**Miền tìm kiếm (Cận dưới / Cận trên)** — áp dụng cho MỌI biến, đóng vai
trò miền xác định của bài toán tối ưu toàn cục (giống `bounds` trong
`scipy.optimize.differential_evolution`): cá thể của GA bị cắt (clip) về
đúng miền này sau mỗi lần lai ghép / đột biến — khác với GA có ràng buộc
(xem `GA_test.ipynb`), nơi hộp chỉ dùng để khởi tạo.

In [2]:
import ipywidgets as W
from IPython.display import clear_output, display

_LBL = {"description_width": "130px"}
_WIDE = W.Layout(width="580px")

w_objective = W.Text(
    value="-cos(x)*cos(y)*exp(-((x-pi)^2+(y-pi)^2))",
    description="Hàm mục tiêu",
    placeholder="ví dụ:  -cos(x)*cos(y)*exp(-((x-pi)^2+(y-pi)^2))",
    layout=_WIDE, style=_LBL, continuous_update=False,
)


# Ba/hai ô tham số nằm chung một hàng — xem GA_test.ipynb: không dùng
# tham số `description` của widget vì bề ngang label đó nằm NGOÀI
# layout.width nên luôn tràn; dùng Label riêng để tính đúng bề ngang.
def _o_tham_so(nhan, gia_tri, kieu=W.IntText):
    o = kieu(value=gia_tri, layout=W.Layout(width="80px"))
    hop = W.HBox(
        [W.Label(nhan, layout=W.Layout(width="90px")), o],
        layout=W.Layout(width="170px"),
    )
    return o, hop


w_lower, _hop_lower = _o_tham_so("Cận dưới", -100.0, W.FloatText)
w_upper, _hop_upper = _o_tham_so("Cận trên", 100.0, W.FloatText)

w_population, _hop_population = _o_tham_so("Quần thể", 500)
w_generations, _hop_generations = _o_tham_so("Số thế hệ", 50)
w_runs, _hop_runs = _o_tham_so("Số lần chạy", 3)

status = W.Output()
apply_button = W.Button(description="Áp dụng bài toán", button_style="primary",
                        icon="check", layout=W.Layout(width="200px"))

problem_ready = False


def _read_form():
    """Đọc form và dựng bài toán. Ném ValueError nếu cú pháp sai."""
    objective_text = w_objective.value.strip()
    return build_unconstrained_problem(objective_text), objective_text


def _on_change(_=None):
    """Gõ xong hàm mục tiêu thì cập nhật danh sách biến ngay."""
    with status:
        clear_output()
        try:
            (expr, found, _raw), _text = _read_form()
        except Exception as error:
            print("✗", error)
            return
        print("Đọc được :", expr)
        print("Biến     :", ", ".join(str(v) for v in found))
        print("\nBấm 'Áp dụng bài toán' để chạy.")


def _apply(_=None):
    global OBJECTIVE, objective_expr, variables, objective, bounds
    global POPULATION_SIZE, GENERATIONS, N_RUNS, problem_ready

    with status:
        clear_output()
        problem_ready = False
        try:
            (objective_expr, variables, objective), OBJECTIVE = _read_form()
        except Exception as error:
            print("✗", error)
            return

        lower = float(w_lower.value)
        upper = float(w_upper.value)

        if not lower < upper:
            print("✗ Cận dưới phải nhỏ hơn cận trên.")
            return

        # Miền tìm kiếm áp dụng CHO MỌI biến — vừa là hộp khởi tạo quần
        # thể, vừa là miền xác định thật sự: cá thể bị clip về đây sau
        # mỗi lần lai ghép / đột biến (xem genetic_algorithm_unconstrained).
        bounds = [(lower, upper)] * len(variables)

        POPULATION_SIZE = int(w_population.value)
        GENERATIONS = int(w_generations.value)
        N_RUNS = int(w_runs.value)
        problem_ready = True

        show_problem(objective_expr, variables, bounds)


w_objective.observe(_on_change, names="value")
w_lower.observe(_on_change, names="value")
w_upper.observe(_on_change, names="value")
apply_button.on_click(_apply)

display(W.VBox([
    W.HTML("<b>Bài toán</b>"),
    w_objective,
    W.HBox([_hop_lower, _hop_upper], layout=W.Layout(width="580px")),
    W.HTML("<b>Tham số GA</b>"),
    W.HBox([_hop_population, _hop_generations, _hop_runs],
           layout=W.Layout(width="580px")),
    apply_button,
    status,
]))

# Áp dụng luôn với giá trị đang có trong form, để "Run All" chạy được ngay.
# Sau khi sửa form thì bấm nút "Áp dụng bài toán" rồi chạy lại từ cell ②.
_apply()


---
## ② Chạy Genetic Algorithm

In [3]:
assert problem_ready, (
    "Bài toán ở cell ① chưa hợp lệ. Xem thông báo lỗi ngay dưới form ở cell ①, "
    "sửa lại rồi bấm nút 'Áp dụng bài toán'."
)

ga_runs = [
    genetic_algorithm_unconstrained(
        objective,
        bounds,
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=42 + i,
    )
    for i in range(N_RUNS)
]

# Lần chạy tốt nhất: f(x) nhỏ nhất trong N_RUNS lần chạy độc lập.
ga_result = min(ga_runs, key=lambda r: r["fun"])

show_solution(
    "GENETIC ALGORITHM — lần chạy tốt nhất",
    ga_result, variables,
)

if N_RUNS > 1:
    show_statistics(ga_runs)


**GENETIC ALGORITHM — lần chạy tốt nhất**

$$\begin{aligned}x &= 3.1415926394 \\ y &= 3.1415926529\end{aligned}$$
$$f^{*} = -1.0000000000$$

| | |
|---|---|
| Thời gian chạy | $1.094093\ \text{s}$ |

**Thống kê GA qua 3 lần chạy độc lập**

| | |
|---|---|
| Tốt nhất | $\min f = -1.0000000000$ |
| Trung bình | $\bar{f} = -1.0000000000$ |
| Tệ nhất | $\max f = -1.0000000000$ |
| Độ lệch chuẩn | $\sigma = 6.8439 \times 10^{-16}$ |

---
## ③ Cài đặt bằng PyGAD (thư viện GA có sẵn, dùng làm mốc so sánh)

In [4]:
assert problem_ready, (
    "Bài toán ở cell ① chưa hợp lệ. Xem thông báo lỗi ngay dưới form ở cell ①, "
    "sửa lại rồi bấm nút 'Áp dụng bài toán'."
)

pygad_runs = [
    run_pygad(
        objective, bounds, len(variables),
        population_size=POPULATION_SIZE,
        generations=GENERATIONS,
        seed=42 + i,
    )
    for i in range(N_RUNS)
]

# Lần chạy tốt nhất: f(x) nhỏ nhất trong N_RUNS lần chạy độc lập.
pygad_result = min(pygad_runs, key=lambda r: r["fun"])

show_solution(
    "PYGAD — lần chạy tốt nhất",
    pygad_result, variables,
)

if N_RUNS > 1:
    show_statistics(pygad_runs)


**PYGAD — lần chạy tốt nhất**

$$\begin{aligned}x &= 3.1415926575 \\ y &= 3.1415926605\end{aligned}$$
$$f^{*} = -1.0000000000$$

| | |
|---|---|
| Thời gian chạy | $1.360878\ \text{s}$ |

**Thống kê GA qua 3 lần chạy độc lập**

| | |
|---|---|
| Tốt nhất | $\min f = -1.0000000000$ |
| Trung bình | $\bar{f} = -1.0000000000$ |
| Tệ nhất | $\max f = -1.0000000000$ |
| Độ lệch chuẩn | $\sigma = 1.0470 \times 10^{-14}$ |

---
## ④ Bảng so sánh

In [5]:
show_comparison(
    [
        ("Genetic Algorithm (tự cài)", ga_result),
        ("PyGAD", pygad_result),
    ],
    variables,
)


| Phương pháp | $f^{*}$ | Thời gian (s) | $x$ | $y$ |
|---|---|---|---|---|
| Genetic Algorithm (tự cài) | $-1.0000000000$ | $1.094093$ | $3.1415926394$ | $3.1415926529$ |
| PyGAD | $-1.0000000000$ | $1.360878$ | $3.1415926575$ | $3.1415926605$ |

---
## ⑤ Kiểm thử trên các hàm benchmark kinh điển

Đặt thủ công (không qua form ①) bốn hàm benchmark kinh điển trong tối ưu
không ràng buộc, mỗi hàm dùng miền tìm kiếm chuẩn hay dùng trong tài
liệu, rồi chạy GA tự cài và PyGAD giống hệt cell ②③ (cùng `POPULATION_SIZE`,
`GENERATIONS`, `N_RUNS` đang đặt ở form ①) và lập bảng so sánh cho từng hàm.

- **Rastrigin** — nhiều cực tiểu cục bộ dày đặc quanh một cực tiểu toàn cục.
- **Rosenbrock** — "thung lũng cong" hẹp, khó dò theo hơn là khó tìm.
- **Ackley** — hố lớn có gợn sóng cực tiểu cục bộ nhỏ phủ lên trên.
- **Schwefel** — hàm "lừa": vùng quanh gốc tọa độ trông tốt nhưng cực tiểu toàn cục thực ra nằm gần biên miền tìm kiếm.

In [6]:
assert problem_ready, (
    "Bài toán ở cell ① chưa hợp lệ (cần để có POPULATION_SIZE/GENERATIONS/"
    "N_RUNS). Xem thông báo lỗi ở cell ①, sửa lại rồi bấm 'Áp dụng bài toán'."
)

x, y = sympy.symbols("x y")
variables_xy = [x, y]

# Ba hàm benchmark kinh điển, đặt thủ công (không qua form/parser) —
# miền tìm kiếm lấy theo giá trị chuẩn thường dùng trong tài liệu.
BENCHMARKS = {
    "Rastrigin": {
        "expr": (
            20 + x**2 - 10 * sympy.cos(2 * sympy.pi * x)
            + y**2 - 10 * sympy.cos(2 * sympy.pi * y)
        ),
        "bounds": [(-5.12, 5.12), (-5.12, 5.12)],
        "optimum": "f(0, 0) = 0",
    },
    "Rosenbrock": {
        "expr": (1 - x) ** 2 + 100 * (y - x**2) ** 2,
        "bounds": [(-5.0, 10.0), (-5.0, 10.0)],
        "optimum": "f(1, 1) = 0",
    },
    "Ackley": {
        "expr": (
            -20 * sympy.exp(-0.2 * sympy.sqrt(0.5 * (x**2 + y**2)))
            - sympy.exp(0.5 * (sympy.cos(2 * sympy.pi * x) + sympy.cos(2 * sympy.pi * y)))
            + 20 + sympy.E
        ),
        "bounds": [(-32.768, 32.768), (-32.768, 32.768)],
        "optimum": "f(0, 0) = 0",
    },
    "Schwefel": {
        "expr": (
            418.9829 * 2
            - x * sympy.sin(sympy.sqrt(sympy.Abs(x)))
            - y * sympy.sin(sympy.sqrt(sympy.Abs(y)))
        ),
        "bounds": [(-500.0, 500.0), (-500.0, 500.0)],
        "optimum": r"f(420.9687, 420.9687) \approx 0",
    },
}

for name, spec in BENCHMARKS.items():

    expr = spec["expr"]
    bounds = spec["bounds"]

    objective_raw = sympy.lambdify(variables_xy, expr, modules="numpy")

    def objective(v):
        try:
            value = float(np.asarray(objective_raw(*v)).reshape(()))
            if np.isfinite(value):
                return value
        except Exception:
            pass
        return np.inf

    display(Markdown(f"### {name} — nghiệm đúng: ${spec['optimum']}$"))
    show_problem(expr, variables_xy, bounds)

    ga_runs = [
        genetic_algorithm_unconstrained(
            objective, bounds,
            population_size=POPULATION_SIZE,
            generations=GENERATIONS,
            seed=42 + i,
        )
        for i in range(N_RUNS)
    ]
    ga_result = min(ga_runs, key=lambda r: r["fun"])

    pygad_runs = [
        run_pygad(
            objective, bounds, len(variables_xy),
            population_size=POPULATION_SIZE,
            generations=GENERATIONS,
            seed=42 + i,
        )
        for i in range(N_RUNS)
    ]
    pygad_result = min(pygad_runs, key=lambda r: r["fun"])

    show_comparison(
        [
            ("Genetic Algorithm (tự cài)", ga_result),
            ("PyGAD", pygad_result),
        ],
        variables_xy,
    )


### Rastrigin — nghiệm đúng: $f(0, 0) = 0$

$$
\underset{x, y}{\text{minimize}} \quad f\left(x, y\right) = x^{2} + y^{2} - 10 \cos{\left(2 \pi x \right)} - 10 \cos{\left(2 \pi y \right)} + 20
$$

Miền tìm kiếm: $-5.12 \le x \le 5.12, \ -5.12 \le y \le 5.12$

| Phương pháp | $f^{*}$ | Thời gian (s) | $x$ | $y$ |
|---|---|---|---|---|
| Genetic Algorithm (tự cài) | $0.0000000000$ | $1.454036$ | $8.0012 \times 10^{-10}$ | $-2.1100 \times 10^{-9}$ |
| PyGAD | $0.0000000000$ | $1.682758$ | $-9.2731 \times 10^{-11}$ | $-2.9847 \times 10^{-9}$ |

### Rosenbrock — nghiệm đúng: $f(1, 1) = 0$

$$
\underset{x, y}{\text{minimize}} \quad f\left(x, y\right) = \left(1 - x\right)^{2} + 100 \left(- x^{2} + y\right)^{2}
$$

Miền tìm kiếm: $-5 \le x \le 10, \ -5 \le y \le 10$

...

$$
\underset{x, y}{\text{minimize}} \quad f\left(x, y\right) = - e^{0.5 \cos{\left(2 \pi x \right)} + 0.5 \cos{\left(2 \pi y \right)}} + e + 20 - 20 e^{- 0.2 \sqrt{0.5 x^{2} + 0.5 y^{2}}}
$$

Miền tìm kiếm: $-32.768 \le x \le 32.768, \ -32.768 \le y \le 32.768$

| Phương pháp | $f^{*}$ | Thời gian (s) | $x$ | $y$ |
|---|---|---|---|---|
| Genetic Algorithm (tự cài) | $5.9089 \times 10^{-11}$ | $1.356361$ | $-1.6166 \times 10^{-11}$ | $-1.3232 \times 10^{-11}$ |
| PyGAD | $2.9921 \times 10^{-11}$ | $2.045572$ | $-1.0552 \times 10^{-11}$ | $7.5011 \times 10^{-13}$ |

### Schwefel — nghiệm đúng: $f(420.9687, 420.9687) \approx 0$

$$
\underset{x, y}{\text{minimize}} \quad f\left(x, y\right) = - x \sin{\left(\sqrt{\left|{x}\right|} \right)} - y \sin{\left(\sqrt{\left|{y}\right|} \right)} + 837.9658
$$

Miền tìm kiếm: $-500 \le x \le 500, \ -500 \le y \le 500$

| Phương pháp | $f^{*}$ | Thời gian (s) | $x$ | $y$ |
|---|---|---|---|---|
| Genetic Algorithm (tự cài) | $2.5455 \times 10^{-5}$ | $1.736427$ | $420.9687461844$ | $420.9687460969$ |
| PyGAD | $2.5455 \times 10^{-5}$ | $2.091201$ | $420.9687463690$ | $420.9687461346$ |